# Pipeline Submission BFGAI - Muslichin

Notebook ini berisi eksperimen image generation, inpainting, outpainting, batch inference, dan perbandingan scheduler sesuai instruksi submission Dicoding Proyek Image Generation.

# Preparing Dependencies

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision Pillow matplotlib numpy

In [ ]:
import gc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from diffusers import (
    DDIMScheduler,
    DPMSolverMultistepScheduler,
    EulerAncestralDiscreteScheduler,
    StableDiffusionImg2ImgPipeline,
    StableDiffusionInpaintPipeline,
    StableDiffusionPipeline,
)
from PIL import Image, ImageDraw, ImageFilter
from transformers import pipeline as transformers_pipeline

BASE_MODEL_ID = "runwayml/stable-diffusion-v1-5"
INPAINT_MODEL_ID = "runwayml/stable-diffusion-inpainting"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

NEGATIVE_PROMPT = (
    "photorealistic, realistic, photograph, 3d render, messy, blurry, "
    "low quality, bad art, ugly, sketch, grainy, unfinished, chromatic aberration"
)
PROMPT = (
    "a dreamy hand drawn sci-fi illustration of a tiny astronaut floating near a colorful planet, "
    "soft pastel nebula, cinematic composition, detailed stars, whimsical digital painting"
)

# Kriteria 1: Melakukan Image Generation dari Teks (Text-to-Image)

## Load Base Pipeline Model

In [ ]:
def flush_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_text_to_image_pipeline():
    pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL_ID, torch_dtype=DTYPE)
    pipe = pipe.to(DEVICE)
    pipe.enable_attention_slicing()
    pipe.enable_vae_slicing()
    return pipe


pipe = load_text_to_image_pipeline()

## Generate Image

In [ ]:
def make_generator(seed):
    return torch.Generator(device=DEVICE).manual_seed(int(seed))


def generate_simple_image(pipe, prompt, negative_prompt, seed):
    generator = make_generator(seed)
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
    )
    return result.images[0]


simple_image = generate_simple_image(pipe, PROMPT, NEGATIVE_PROMPT, seed=222)
plt.figure(figsize=(6, 6))
plt.imshow(simple_image)
plt.axis("off")
plt.title("generate_simple_image - seed 222")
plt.show()

## Generate Image with Hyperparameter Configuration

In [ ]:
def generate_advanced_image(pipe, prompt, negative_prompt, seed, guidance_scale, num_inference_step):
    generator = make_generator(seed)
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
        guidance_scale=float(guidance_scale),
        num_inference_steps=int(num_inference_step),
    )
    return result.images[0]


advanced_image = generate_advanced_image(
    pipe,
    PROMPT,
    NEGATIVE_PROMPT,
    seed=222,
    guidance_scale=9.0,
    num_inference_step=40,
)
plt.figure(figsize=(6, 6))
plt.imshow(advanced_image)
plt.axis("off")
plt.title("generate_advanced_image - seed 222")
plt.show()

## Guidance Scale Comparison

In [ ]:
guidance_results = []
for cfg in [3.0, 12.0]:
    image = generate_advanced_image(
        pipe,
        PROMPT,
        NEGATIVE_PROMPT,
        seed=222,
        guidance_scale=cfg,
        num_inference_step=35,
    )
    guidance_results.append((cfg, image))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (cfg, image) in zip(axes, guidance_results):
    ax.imshow(image)
    ax.set_title(f"Guidance Scale {cfg}")
    ax.axis("off")
plt.show()

### Guidance Scale Explanation:

* **Gambar dengan Scale Rendah:** hasil biasanya lebih longgar terhadap prompt. Komposisi dapat terlihat lebih variatif, tetapi beberapa detail spesifik dari prompt mungkin tidak muncul dengan kuat.
* **Gambar dengan Scale Tinggi:** hasil biasanya lebih patuh terhadap prompt dan detail utama lebih tegas, tetapi jika terlalu tinggi gambar dapat terlihat kurang natural atau memiliki artefak.

## Inference Steps Comparison

In [ ]:
step_results = []
for steps in [10, 45]:
    image = generate_advanced_image(
        pipe,
        PROMPT,
        NEGATIVE_PROMPT,
        seed=222,
        guidance_scale=8.0,
        num_inference_step=steps,
    )
    step_results.append((steps, image))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (steps, image) in zip(axes, step_results):
    ax.imshow(image)
    ax.set_title(f"Inference Steps {steps}")
    ax.axis("off")
plt.show()

### Inference Step Explanation:

* **Gambar dengan Step Rendah:** proses lebih cepat, tetapi detail dan ketajaman biasanya lebih rendah. Noise, bentuk yang belum stabil, atau tekstur yang kurang halus lebih mudah terlihat.
* **Gambar dengan Step Tinggi:** hasil biasanya lebih halus, detail lebih matang, dan struktur visual lebih stabil. Waktu inference lebih lama, sehingga perlu disesuaikan dengan kapasitas GPU.

## Batch Inference from One Prompt

In [ ]:
def generate_batch_images(pipe, prompt, negative_prompt, seed, guidance_scale, num_inference_steps, num_images=4):
    generators = [make_generator(seed + idx) for idx in range(num_images)]
    result = pipe(
        prompt=[prompt] * num_images,
        negative_prompt=[negative_prompt] * num_images,
        generator=generators,
        guidance_scale=float(guidance_scale),
        num_inference_steps=int(num_inference_steps),
    )
    return result.images


batch_images = generate_batch_images(pipe, PROMPT, NEGATIVE_PROMPT, 222, 8.0, 35, num_images=4)
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for idx, ax in enumerate(axes.ravel()):
    ax.imshow(batch_images[idx])
    ax.set_title(f"Image {idx + 1}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Load Scheduler

In [ ]:
def load_scheduler(pipe, scheduler_name):
    scheduler_name = scheduler_name.strip().lower()
    if scheduler_name == "euler a":
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    elif scheduler_name == "dpm++":
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
    elif scheduler_name == "ddim":
        pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    else:
        raise ValueError("scheduler_name must be one of: Euler A, DPM++, DDIM")
    return pipe


scheduler_results = []
for scheduler_name in ["Euler A", "DPM++", "DDIM"]:
    pipe = load_scheduler(pipe, scheduler_name)
    image = generate_advanced_image(pipe, PROMPT, NEGATIVE_PROMPT, 222, 8.0, 35)
    scheduler_results.append((scheduler_name, image))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (scheduler_name, image) in zip(axes, scheduler_results):
    ax.imshow(image)
    ax.set_title(scheduler_name)
    ax.axis("off")
plt.show()

### Scheduler Comparation:

* **Gambar dengan Euler A Scheduler:** variasi visual biasanya lebih ekspresif dan tekstur dapat terasa artistik, cocok untuk eksplorasi cepat.
* **Gambar dengan DPM++ Scheduler:** hasil cenderung stabil, detail lebih rapi, dan kualitas baik pada jumlah step sedang.
* **Gambar dengan DDIM Scheduler:** hasil umumnya konsisten dan deterministic, tetapi dapat terlihat lebih sederhana dibanding scheduler lain pada konfigurasi tertentu.

# Kriteria 2: Menyempurnakan Gambar Melalui Image-to-Image

## Base + Refiner Image Generation

In [ ]:
def two_stage_generation(base_pipe, prompt, negative_prompt, seed, guidance_scale=8.0, steps=40):
    # Stage 1 menggunakan base text-to-image untuk menghasilkan rancangan awal.
    base_image = generate_advanced_image(
        base_pipe,
        prompt,
        negative_prompt,
        seed=seed,
        guidance_scale=guidance_scale,
        num_inference_step=max(10, int(steps * 0.8)),
    )

    # Stage 2 menggunakan img2img sebagai refiner ringan dengan model yang sama agar tetap hemat VRAM.
    refiner = StableDiffusionImg2ImgPipeline.from_pretrained(BASE_MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
    refiner.enable_attention_slicing()
    refined = refiner(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=base_image,
        strength=0.25,
        guidance_scale=guidance_scale,
        num_inference_steps=max(8, int(steps * 0.2)),
        generator=make_generator(seed + 1000),
    ).images[0]
    del refiner
    flush_memory()
    return base_image, refined


base_stage_image, refined_image = two_stage_generation(pipe, PROMPT, NEGATIVE_PROMPT, seed=222)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, title, image in zip(axes, ["Base", "Refined"], [base_stage_image, refined_image]):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
plt.show()

## Inpainting

### Load Model Inpainting

In [ ]:
def load_inpainting_pipeline():
    pipe_inpaint = StableDiffusionInpaintPipeline.from_pretrained(INPAINT_MODEL_ID, torch_dtype=DTYPE)
    pipe_inpaint = pipe_inpaint.to(DEVICE)
    pipe_inpaint.enable_attention_slicing()
    pipe_inpaint.enable_vae_slicing()
    return pipe_inpaint


pipe_inpaint = load_inpainting_pipeline()

### Manual Masking

In [ ]:
def create_manual_mask(image, box_ratio=(0.58, 0.25, 0.9, 0.58)):
    width, height = image.size
    x1 = int(width * box_ratio[0])
    y1 = int(height * box_ratio[1])
    x2 = int(width * box_ratio[2])
    y2 = int(height * box_ratio[3])

    mask = Image.new("L", image.size, 0)
    draw = ImageDraw.Draw(mask)
    draw.ellipse((x1, y1, x2, y2), fill=255)
    return mask.filter(ImageFilter.GaussianBlur(radius=4))


manual_mask = create_manual_mask(advanced_image)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(advanced_image)
axes[0].set_title("Source")
axes[0].axis("off")
axes[1].imshow(manual_mask, cmap="gray")
axes[1].set_title("Manual Mask")
axes[1].axis("off")
plt.show()

### Generate

In [ ]:
def inpaint_engine(pipe, image, mask, prompt, seed=9, negative_prompt=NEGATIVE_PROMPT, strength=0.9, guidance_scale=8.0, steps=35):
    if image.mode != "RGB":
        image = image.convert("RGB")
    if mask.mode != "L":
        mask = mask.convert("L")
    if mask.size != image.size:
        mask = mask.resize(image.size, resample=Image.NEAREST)

    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image,
        mask_image=mask,
        strength=float(strength),
        guidance_scale=float(guidance_scale),
        num_inference_steps=int(steps),
        generator=make_generator(seed),
    )
    return result.images[0]


inpaint_prompt = "a broken satellite floating in space, detailed debris, illustrated style, matching lighting"
inpainted_image = inpaint_engine(pipe_inpaint, advanced_image, manual_mask, inpaint_prompt, seed=9)
plt.figure(figsize=(6, 6))
plt.imshow(inpainted_image)
plt.axis("off")
plt.title("Manual Inpainting - seed 9")
plt.show()

## Inpainting Menggunakan Automasking

### Load Model Segmentation Untuk Masking

In [ ]:
# Model segmentation digunakan untuk membuat mask otomatis. Jalankan bagian ini bila runtime memiliki VRAM/RAM yang cukup.
mask_generator = transformers_pipeline("mask-generation", model="facebook/sam-vit-base", device=0 if DEVICE == "cuda" else -1)

### Masking with Segmentation Model

In [ ]:
def create_auto_mask(image, min_area_ratio=0.03):
    masks = mask_generator(image)
    width, height = image.size
    min_area = width * height * min_area_ratio

    selected = None
    for item in masks.get("masks", []):
        arr = np.asarray(item).astype(np.uint8)
        if arr.sum() >= min_area:
            selected = arr
            break

    if selected is None:
        return create_manual_mask(image)

    mask = Image.fromarray((selected * 255).astype(np.uint8), mode="L")
    return mask.resize(image.size, resample=Image.NEAREST).filter(ImageFilter.MaxFilter(9))


auto_mask = create_auto_mask(advanced_image)
plt.figure(figsize=(5, 5))
plt.imshow(auto_mask, cmap="gray")
plt.axis("off")
plt.title("Automatic Segmentation Mask")
plt.show()

### Generate

In [ ]:
auto_inpainted_image = inpaint_engine(pipe_inpaint, advanced_image, auto_mask, inpaint_prompt, seed=9)
plt.figure(figsize=(6, 6))
plt.imshow(auto_inpainted_image)
plt.axis("off")
plt.title("Automatic Mask Inpainting")
plt.show()

## Outpainting

### Prepare the Canvas

In [ ]:
def prepare_outpainting(image, direction="right", expand_pixels=128):
    if image.mode != "RGB":
        image = image.convert("RGB")

    width, height = image.size
    direction = direction.lower()
    left = expand_pixels if direction == "left" else 0
    right = expand_pixels if direction == "right" else 0
    top = expand_pixels if direction == "top" else 0
    bottom = expand_pixels if direction == "bottom" else 0

    new_width = width + left + right
    new_height = height + top + bottom
    new_width -= new_width % 8
    new_height -= new_height % 8

    bg = image.resize((new_width, new_height), resample=Image.BICUBIC).filter(ImageFilter.GaussianBlur(radius=40))
    canvas = bg.copy()
    paste_x = left
    paste_y = top
    canvas.paste(image.resize((width, height)), (paste_x, paste_y))

    mask = Image.new("L", (new_width, new_height), 255)
    keep = Image.new("L", (width, height), 0)
    mask.paste(keep, (paste_x, paste_y))
    return canvas, mask


outpaint_canvas, outpaint_mask = prepare_outpainting(inpainted_image, direction="right", expand_pixels=128)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(outpaint_canvas)
axes[0].set_title("Canvas")
axes[0].axis("off")
axes[1].imshow(outpaint_mask, cmap="gray")
axes[1].set_title("Mask")
axes[1].axis("off")
plt.show()

### Generate

In [ ]:
outpaint_prompt = "wide view of a dreamy illustrated astronaut scene in space with planets, stars, and a broken satellite"
outpainted_image = inpaint_engine(pipe_inpaint, outpaint_canvas, outpaint_mask, outpaint_prompt, seed=9, strength=1.0)
plt.figure(figsize=(7, 6))
plt.imshow(outpainted_image)
plt.axis("off")
plt.title("Outpainting One Side")
plt.show()

## Outpainting Zoom Out

### Prepare Canvas for Zoom Out

In [ ]:
def prepare_zoom_out(image, expand_pixels=128):
    if image.mode != "RGB":
        image = image.convert("RGB")

    width, height = image.size
    new_width = width + (expand_pixels * 2)
    new_height = height + (expand_pixels * 2)
    new_width -= new_width % 8
    new_height -= new_height % 8

    bg = image.resize((new_width, new_height), resample=Image.BICUBIC).filter(ImageFilter.GaussianBlur(radius=45))
    canvas = bg.copy()
    paste_x = (new_width - width) // 2
    paste_y = (new_height - height) // 2
    canvas.paste(image, (paste_x, paste_y))

    mask = Image.new("L", (new_width, new_height), 255)
    mask.paste(Image.new("L", (width, height), 0), (paste_x, paste_y))
    return canvas, mask


zoom_canvas, zoom_mask = prepare_zoom_out(outpainted_image, expand_pixels=128)

### Generate

In [ ]:
zoomed_out_image = inpaint_engine(pipe_inpaint, zoom_canvas, zoom_mask, outpaint_prompt, seed=9, strength=1.0)
plt.figure(figsize=(7, 6))
plt.imshow(zoomed_out_image)
plt.axis("off")
plt.title("Zoom Out Outpainting")
plt.show()

flush_memory()